In [ ]:
discard_electrodes = [
    "E1", "E8", "E14", "E17", "E21", "E25", "E32", "E38", "E43", "E44", 
    "E48", "E49", "E56", "E63", "E68", "E94", "E99", "E107", "E113", "E114",
    "E119", "E120", "E121", "E125", "E126", "E127", "E128", "E73", "E81", "E88"
]


# Baseline time period testing:

In [4]:

import os
import glob
import mne
import numpy as np

def read_any_raw(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.raw', '.mff'):
        return mne.io.read_raw_egi(path, preload=True)


def preprocess_raw(raw, notch_freq=50., l_freq=1., h_freq=64., montage_name='GSN-HydroCel-129'):
    discard_electrodes = [
        "E1", "E8", "E14", "E17", "E21", "E25", "E32", "E38", "E43", "E44", 
        "E48", "E49", "E56", "E63", "E68", "E94", "E99", "E107", "E113", "E114",
        "E119", "E120", "E121", "E125", "E126", "E127", "E128", "E73", "E81", "E88"
    ]

    try:
        mont = mne.channels.make_standard_montage(montage_name)
        raw.set_montage(mont, match_case=False)
    except Exception:
        pass

    # Drop reference if it exists
    if 'VREF' in raw.ch_names:
        raw.drop_channels('VREF')

    # mark bads instead of dropping them
    raw.info['bads'] = [ch for ch in discard_electrodes if ch in raw.ch_names]

    # filter
    raw.notch_filter(freqs=notch_freq, picks='eeg', verbose=False)
    raw.filter(l_freq=l_freq, h_freq=h_freq, picks='eeg', verbose=False)

    return raw
def chunk_epochs(epochs, n_chunks=3):
    sfreq = epochs.info['sfreq']
    n_times = len(epochs.times)
    chunk_len = n_times // n_chunks

    X = epochs.get_data()  # shape: (n_epochs, n_channels, n_times)
    y = epochs.events[:, -1]

    X_chunks, y_chunks = [], []

    for i in range(n_chunks):
        start = i * chunk_len
        stop = (i + 1) * chunk_len
        X_chunk = X[:, :, start:stop]
        X_chunks.append(X_chunk)
        y_chunks.append(y)  # label same as parent epoch

    X_new = np.concatenate(X_chunks, axis=0)
    y_new = np.concatenate(y_chunks, axis=0)
    print(f"Chunked: {X.shape[0]} → {X_new.shape[0]} samples, each {X_new.shape[2]/sfreq:.2f}s long")
    return X_new, y_new

def extract_start_event_ids(event_id_map):
    # select annotation keys that mark START of Observation/Imagination (OS*, IS*)
    return {k: v for k, v in event_id_map.items() if k.startswith('OS') or k.startswith('IS')}
def epoch_recording(path, out_dir, tmin=0.0, tmax=4.0, reject=None):
    raw = read_any_raw(path)
    raw.pick_types(eeg=True)

    raw = preprocess_raw(raw)

    if raw.info["bads"]:
        raw.drop_channels(raw.info["bads"])
        print(f"Dropped {len(raw.info['bads'])} bad channels: {raw.info['bads']}")

    # --- Define fixed canonical mapping for all recordings ---
    CANONICAL_EVENT_ID = {
        'ISBA': 0, 'ISBY': 1, 'ISDO': 2, 'ISMO': 3, 'ISSI': 4,
        'OSBA': 10, 'OSBY': 11, 'OSDO': 12, 'OSMO': 13, 'OSSI': 14,
    }

    # --- Manually extract canonical events from annotations ---
    annots = raw.annotations
    events_list = []
    for a in annots:
        desc = a['description'].strip().upper()
        if desc in CANONICAL_EVENT_ID:
            sample = int(a['onset'] * raw.info['sfreq'])
            events_list.append([sample, 0, CANONICAL_EVENT_ID[desc]])

    if not events_list:
        print(f"No canonical events found in {os.path.basename(path)}")
        return None

    events = np.array(events_list, dtype=int)
    event_id = CANONICAL_EVENT_ID
    sel_event_id = extract_start_event_ids(event_id)

    # --- Align relative to baseline END (BSEN) ---
    bs_end = [a['onset'] for a in annots if 'BSEN' in a['description'].upper()]
    if bs_end:
        offset = bs_end[0]
        events[:, 0] -= int(offset * raw.info['sfreq'])
        print(f"⏱️ Re-aligned events in {os.path.basename(path)} by {-offset:.2f}s (BSEN)")

    # --- Create epochs ---
    epochs = mne.Epochs(
        raw, events, event_id=sel_event_id,
        tmin=tmin, tmax=tmax,
        baseline=None, preload=True, reject=reject, verbose=False
    )
        # 💡 ADD THIS BLOCK HERE --------------------------------
    X_chunked, y_chunked = chunk_epochs(epochs, n_chunks=3)
    np.save(os.path.join(out_dir, "X_chunked.npy"), X_chunked)
    np.save(os.path.join(out_dir, "y_chunked.npy"), y_chunked)
    print(f"✅ Saved chunked arrays to {out_dir} with shape {X_chunked.shape}")
    # --------
    out_path = os.path.join(out_dir, f"{os.path.splitext(os.path.basename(path))[0]}-epo.fif")
    os.makedirs(out_dir, exist_ok=True)
    epochs.save(out_path, overwrite=True)
    return epochs



# Run over Data folder: assumes structure Data/<subject>/*.raw|*.mff|*.edf|*.fif
DATA_DIR = "/Users/kavinfidel/Documents/Fidel/CNS_Lab/VM_EEG/Data"
OUT_DIR = "/Users/kavinfidel/Documents/Fidel/CNS_Lab/VM_EEG/epochs_bads_drop_chunks"

for subj_dir in sorted(glob.glob(os.path.join(DATA_DIR, "*"))):
    if not os.path.isdir(subj_dir): 
        continue
    rec_files = []
    for ext in ('*.raw','*.mff','*.edf','*.fif','*.edf.gz','*.raw.gz'):
        rec_files.extend(glob.glob(os.path.join(subj_dir, ext)))
    for rec in sorted(rec_files):
        try:
            epoch_recording(rec, out_dir=os.path.join(OUT_DIR, os.path.basename(subj_dir)))
        except Exception as e:
            print(f"Error processing {rec}: {e}")


Dropped 0 bad channels: []
⏱️ Re-aligned events in VI_S1_S1_B1_1700_08102025_20251008_050540.mff by -21.26s (BSEN)
Chunked: 30 → 90 samples, each 1.33s long
Error processing /Users/kavinfidel/Documents/Fidel/CNS_Lab/VM_EEG/Data/S1/VI_S1_S1_B1_1700_08102025_20251008_050540.mff: [Errno 2] No such file or directory: '/Users/kavinfidel/Documents/Fidel/CNS_Lab/VM_EEG/epochs_bads_drop_chunks/S1/X_chunked.npy'
Dropped 0 bad channels: []
⏱️ Re-aligned events in VI_S1_S1_B2_1710_081025_20251008_051501.mff by -21.32s (BSEN)
Chunked: 30 → 90 samples, each 1.33s long
Error processing /Users/kavinfidel/Documents/Fidel/CNS_Lab/VM_EEG/Data/S1/VI_S1_S1_B2_1710_081025_20251008_051501.mff: [Errno 2] No such file or directory: '/Users/kavinfidel/Documents/Fidel/CNS_Lab/VM_EEG/epochs_bads_drop_chunks/S1/X_chunked.npy'


KeyboardInterrupt: 

In [ ]:
import glob, os, mne, numpy as np

In [2]:
mne.set_log_level('ERROR') 
EPOCHS_DIR = "/Users/kavinfidel/Documents/Fidel/CNS_Lab/VM_EEG/epochs_bads_drop_chunks"

epochs_all, labels_all = [], []

for subj_dir in sorted(glob.glob(os.path.join(EPOCHS_DIR, "S*"))):
    X_path = os.path.join(subj_dir, "X_chunked.npy")
    y_path = os.path.join(subj_dir, "y_chunked.npy")

    if not os.path.exists(X_path) or not os.path.exists(y_path):
        print(f"⚠️ Missing chunked data in {subj_dir}")
        continue

    X = np.load(X_path)
    y = np.load(y_path)

    print(f"{os.path.basename(subj_dir)} -> {X.shape}, {y.shape}")
    epochs_all.append(X)
    labels_all.append(y)

X = np.concatenate(epochs_all, axis=0)
y = np.concatenate(labels_all, axis=0)
print("✅ Combined shape:", X.shape, y.shape)

ValueError: need at least one array to concatenate

In [4]:
# imagination and observation label groups
imagination_codes = [0,1,2,3,4]
observation_codes = [10, 11, 12, 13, 14]

# Boolean masks
mask_imag = np.isin(y, imagination_codes)
mask_obs = np.isin(y, observation_codes)

# Separate datasets
X_imag, y_imag = X[mask_imag], y[mask_imag]
X_obs, y_obs = X[mask_obs], y[mask_obs]



* 0 : Banana
* 1 : Baby
* 2 : Dog
* 3 : Monkey
* 4 : Sitar

In [5]:
len(y_obs)

240

In [6]:
len(y_imag)

240

In [7]:
type(y_imag)

numpy.ndarray

In [ ]:
type(X_imag)

In [8]:
SAVE_DIR = "/Users/kavinfidel/Documents/Fidel/CNS_Lab/VM_EEG/processed_data_bads_drop"
os.makedirs(SAVE_DIR, exist_ok=True)

# assume these are your arrays
# X = np array of shape (480, 128, 2001)
# y = np array of shape (480,)

np.save(os.path.join(SAVE_DIR, "X_imag.npy"), X_imag)
np.save(os.path.join(SAVE_DIR, "y_imag.npy"), y_imag)

print("✅ Saved EEG data successfully!")

✅ Saved EEG data successfully!


In [9]:
np.save(os.path.join(SAVE_DIR, "X_obs.npy"), X_obs)
np.save(os.path.join(SAVE_DIR, "y_obs.npy"), y_obs)

print("✅ Saved EEG data successfully!")

✅ Saved EEG data successfully!


In [9]:
ximag = np.load("/Users/kavinfidel/Documents/Fidel/CNS_Lab/VM_EEG/processed_data/y_imag.npy")

In [10]:
ximag.shape

(240,)

In [7]:
16*15

240

In [8]:
240*3

720

In [11]:
# Load your existing numpy arrays
X = np.load("/Users/kavinfidel/Documents/Fidel/CNS_Lab/VM_EEG/processed_data/X_imag.npy")   # (240, 128, 2001)
y = np.load("/Users/kavinfidel/Documents/Fidel/CNS_Lab/VM_EEG/processed_data/y_imag.npy")   # (240,)

# Parameters
chunk_size = 667   # roughly 2001 / 3 → ~1.33s chunks
n_chunks = X.shape[-1] // chunk_size

# Non-overlapping chunking
X_chunked = []
y_chunked = []

for i in range(X.shape[0]):
    for j in range(n_chunks):
        start = j * chunk_size
        end = start + chunk_size
        X_chunked.append(X[i, :, start:end])
        y_chunked.append(y[i])

X_chunked = np.array(X_chunked)
y_chunked = np.array(y_chunked)

print(f"Chunked: {X.shape[0]} → {X_chunked.shape[0]} samples, each {chunk_size/X.shape[-1]*4:.2f}s long")
print(f"New shapes: X {X_chunked.shape}, y {y_chunked.shape}")

# Optional: save them
os.makedirs("chunked_numpy", exist_ok=True)
np.save("chunked_numpy/X_chunked.npy", X_chunked)
np.save("chunked_numpy/y_chunked.npy", y_chunked)

Chunked: 240 → 720 samples, each 1.33s long
New shapes: X (720, 128, 667), y (720,)


In [12]:
X = np.load("/Users/kavinfidel/Documents/Fidel/CNS_Lab/VM_EEG/processed_data/X_obs.npy")   # (240, 128, 2001)
y = np.load("/Users/kavinfidel/Documents/Fidel/CNS_Lab/VM_EEG/processed_data/y_obs.npy")   # (240,)

# Parameters
chunk_size = 667   # roughly 2001 / 3 → ~1.33s chunks
n_chunks = X.shape[-1] // chunk_size

# Non-overlapping chunking
X_chunked = []
y_chunked = []

for i in range(X.shape[0]):
    for j in range(n_chunks):
        start = j * chunk_size
        end = start + chunk_size
        X_chunked.append(X[i, :, start:end])
        y_chunked.append(y[i])

X_chunked = np.array(X_chunked)
y_chunked = np.array(y_chunked)

print(f"Chunked: {X.shape[0]} → {X_chunked.shape[0]} samples, each {chunk_size/X.shape[-1]*4:.2f}s long")
print(f"New shapes: X {X_chunked.shape}, y {y_chunked.shape}")

# Optional: save them
os.makedirs("chunked_numpy", exist_ok=True)
np.save("chunked_numpy/X_chunked_obs.npy", X_chunked)
np.save("chunked_numpy/y_chunked_obs.npy", y_chunked)

Chunked: 240 → 720 samples, each 1.33s long
New shapes: X (720, 128, 667), y (720,)


In [13]:
X_chunked

array([[[ 2.08309128e-05,  2.24018470e-05,  2.20908524e-05, ...,
          2.46526267e-05,  2.70168734e-05,  2.93543887e-05],
        [ 3.35978508e-05,  3.74175863e-05,  3.95701430e-05, ...,
          1.09502164e-05,  1.15417645e-05,  1.29411051e-05],
        [ 2.69211335e-05,  2.82675404e-05,  2.87471550e-05, ...,
         -3.52741947e-06, -2.44685293e-06, -7.64596861e-07],
        ...,
        [-2.03437280e-05, -1.67086650e-05, -1.52752001e-05, ...,
          2.00476991e-05,  2.32947927e-05,  2.70396720e-05],
        [-4.05949742e-05, -3.86428497e-05, -3.60124674e-05, ...,
         -4.63473337e-06, -4.01969491e-06, -4.77579639e-06],
        [-3.47085357e-05, -3.32715367e-05, -3.11140455e-05, ...,
         -1.07103036e-05, -8.60827150e-06, -7.34672927e-06]],

       [[ 3.10789836e-05,  3.20124465e-05,  3.23901785e-05, ...,
         -8.58485008e-06, -6.78039643e-06, -6.26909915e-06],
        [ 1.49729585e-05,  1.71150817e-05,  1.88348412e-05, ...,
          1.92050715e-05,  2.12487539e

In [14]:
print("X shape:", X_chunked.shape)
print("y shape:", y_chunked.shape)
print("X dtype:", X_chunked.dtype, "y dtype:", y_chunked.dtype)
print("X min/max:", X_chunked.min(), X_chunked.max())
print("Unique labels:", np.unique(y_chunked))
print("Any NaN?", np.isnan(X_chunked).any())

X shape: (720, 128, 667)
y shape: (720,)
X dtype: float64 y dtype: int32
X min/max: -0.0014125774462445811 0.0021921788944061493
Unique labels: [10 11 12 13 14]
Any NaN? False
